# DTE Framework — Million-Scale Saturation Attack

**Version**: 3.2.0  
**Environment**: Kaggle GPU/CPU  
**Target**: 1M+ samples across Theorems 1-4

This notebook runs the DTE saturation attack at scale.

In [ ]:
# Install DTE
!pip install -q numpy scipy matplotlib
# If dte-core is on PyPI:
# !pip install -q dte-core

# Or clone from GitHub:
# !git clone https://github.com/chepin-ai/DTE-Project.git
# import sys
# sys.path.insert(0, '/kaggle/working/DTE-Project/python')

In [ ]:
import numpy as np
import json
import time
from concurrent.futures import ProcessPoolExecutor

from dte.core import DTECoreEngine
from dte.states import StateGenerator

print("DTE Framework v3.2.0 — Batch Saturation")

In [ ]:
# Configuration
TOTAL_SAMPLES = 1_000_000
DIMS = [(2,2), (3,3), (4,4), (5,5), (6,6), (7,7), (8,8), (9,9), (10,10)]
BATCH_SIZE = 5000
N_WORKERS = 4  # Adjust based on Kaggle CPU cores

samples_per_dim = TOTAL_SAMPLES // len(DIMS)
print(f"Target: {TOTAL_SAMPLES} samples across {len(DIMS)} dimensions")
print(f"Samples per dimension: {samples_per_dim}")

In [ ]:
def validate_dimension(args):
    """Worker function for parallel validation."""
    da, db, n = args
    d = min(da, db)
    eng = DTECoreEngine(da, db, validate=False)
    c_d = 8.0 * np.log2(d) / ((d-1)**2) if d > 1 else 8.0
    
    t1_pass = 0
    t1_max = 0.0
    t2_pass = 0
    t2_min = float('inf')
    
    for i in range(n):
        seed = i + da * 1000000
        r = i % 4
        if r == 0:
            rho = StateGenerator.random_mixed_state(da*db, seed=seed)
        elif r == 1:
            rho = StateGenerator.random_pure_state(da*db, seed=seed)
        elif r == 2:
            rho = StateGenerator.werner_state(0.3 + 0.4*(i%10)/10, d)
        else:
            rho = StateGenerator.maximally_entangled(d)
        
        # Theorem 1: G = O
        diff = abs(eng.G(rho) - eng.O(rho))
        t1_max = max(t1_max, diff)
        if diff < 1e-7:
            t1_pass += 1
        
        # Theorem 2: I >= cG^2
        t = eng.triple(rho)
        if t.G > 1e-10:
            ratio = t.I / (t.G**2)
            t2_min = min(t2_min, ratio)
            if ratio >= c_d - 1e-6:
                t2_pass += 1
        else:
            t2_pass += 1
    
    return {
        'dims': f'{da}x{db}',
        't1_pass': t1_pass, 't1_total': n, 't1_max': t1_max,
        't2_pass': t2_pass, 't2_total': n, 't2_min': t2_min if t2_min != float('inf') else 0.0
    }

# Build work units
work = [(da, db, samples_per_dim) for da, db in DIMS]

start = time.time()
with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    results = list(executor.map(validate_dimension, work))
elapsed = time.time() - start

for r in results:
    print(f"{r['dims']}: T1={r['t1_pass']}/{r['t1_total']} T2={r['t2_pass']}/{r['t2_total']}")

t1_total = sum(r['t1_pass'] for r in results)
t1_all = sum(r['t1_total'] for r in results)
t2_total = sum(r['t2_pass'] for r in results)
t2_all = sum(r['t2_total'] for r in results)

print(f"\nTotal: T1={t1_total}/{t1_all} ({t1_total/t1_all*100:.2f}%) T2={t2_total}/{t2_all} ({t2_total/t2_all*100:.2f}%)")
print(f"Time: {elapsed:.1f}s")

In [ ]:
# Save results
output = {
    'version': '3.2.0',
    'total_samples': t1_all + t2_all,
    'theorem1': {'pass': t1_total, 'total': t1_all, 'rate': t1_total/t1_all},
    'theorem2': {'pass': t2_total, 'total': t2_all, 'rate': t2_total/t2_all},
    'by_dimension': results,
    'time': elapsed
}

with open('/kaggle/working/dte_million_scale.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Saved to /kaggle/working/dte_million_scale.json")